# rgcn / lsep / relation / n08

```
run      rgcn-lsep-relation-n08
purpose  normalization / per relation
bucket   exapt-lab/hangar/runs/rgcn-lsep-relation-n08
```

In [1]:
REPOSITORY = "https://github.com/aekn/exapt.git"
GIT_SHA = "cda583e5a90386f67f03e1208d14ab00f7405d6d"
DATA_SHA = "a29db6bb7019eefa4fd93d7b771f212cc88014a1"

RUN = "rgcn-lsep-relation-n08"
PURPOSE = "normalization / per relation"

OBJECTIVE = "lsep"
NORMALIZATION = "relation"
NEGATIVES = 8

WIDTH = 128
SEED = 101
EPOCHS = 120

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
DROPOUT = 0.0

In [2]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata


ROOT = Path("/content/exapt")
OUTPUT = Path("/content/runs") / RUN
PYTHON = ROOT / ".venv/bin/python"
HF = ROOT / ".venv/bin/hf"

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)

if not (ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--quiet", REPOSITORY, str(ROOT)], check=True)

subprocess.run(["git", "-C", str(ROOT), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--quiet", "--detach", GIT_SHA], check=True)
subprocess.run(["uv", "python", "install", "3.12", "--quiet"], check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "train", "--no-dev", "--python", "3.12", "--quiet"], cwd=ROOT, check=True)

CompletedProcess(args=['uv', 'sync', '--locked', '--extra', 'train', '--no-dev', '--python', '3.12', '--quiet'], returncode=0)

In [3]:
def train(epochs: int) -> None:
    token = userdata.get("exapt_hf")
    if not token:
        raise RuntimeError("missing Colab secret: exapt_hf")

    env = {
        **os.environ,
        "HF_TOKEN": token,
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    }

    print("source   fetch primekg", flush=True)

    snapshot = subprocess.check_output(
        [
            str(HF), "download", "exapt-lab/data",
            "sources/primekg-2.1/manifest.json",
            "sources/primekg-2.1/kg.csv",
            "--repo-type", "dataset",
            "--revision", DATA_SHA,
            "--quiet",
        ],
        cwd=ROOT,
        env=env,
        text=True,
    ).strip()
    source = Path(snapshot) / "sources/primekg-2.1"
    print("source   ready", flush=True)

    command = [
        str(PYTHON), "-u", "scripts/fit.py", str(source), str(OUTPUT),
        "--encoder", "rgcn",
        "--width", str(WIDTH),
        "--normalization", NORMALIZATION,
        "--layout", "csr",
        "--features", "trainable",
        "--dropout", str(DROPOUT),
        "--objective", OBJECTIVE,
        "--candidate-pool", "sampled",
        "--seed", str(SEED),
        "--epochs", str(epochs),
        "--negatives", str(NEGATIVES),
        "--negative-weight", "1",
        "--learning-rate", str(LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--scheduler", "constant",
        "--benchmark-seed", "163",
        "--folds", "4",
        "--development-fraction", "0.1",
        "--test-fraction", "0.1",
        "--k", "20",
        "--device", "cuda",
        "--dataset-revision", DATA_SHA,
        "--purpose", PURPOSE,
        "--bucket-run", RUN,
    ]
    if (OUTPUT / "resume.pt").is_file():
        command.append("--resume")

    print("train    launch", flush=True)

    with subprocess.Popen(
        command,
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        if process.stdout is None:
            raise RuntimeError("training output pipe unavailable")
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()

    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

In [4]:
train(EPOCHS)

source   fetch primekg
source   ready
train    launch
exapt / rgcn-lsep-relation-n08

model    rgcn / distmult
train    lsep / relation / n8 / w128
seed     101
folds    4
budget   120 epochs
device   cuda / NVIDIA L4
local    /content/runs/rgcn-lsep-relation-n08
bucket   exapt-lab/hangar/runs/rgcn-lsep-relation-n08

source   load therapeutics
source   ready / 17,080 diseases
graph    load source arcs
graph    ready / 8,100,498 arcs
folds    prepare masked contexts

epoch       loss         lr      R@20   nDCG@20       wall
 0001     3.0169   1.00e-03    0.0813    0.0535   00:00:07   *
 0002     2.3236   1.00e-03    0.1108    0.0714   00:00:06   *
 0003     1.9055   1.00e-03    0.1002    0.0662   00:00:06
 0004     1.6588   1.00e-03    0.1267    0.0798   00:00:06   *
 0005     1.4526   1.00e-03    0.1567    0.0935   00:00:06   *
 0006     1.2809   1.00e-03    0.1720    0.1049   00:00:06   *
 0007     1.2398   1.00e-03    0.1914    0.1187   00:00:06   *
 0008     1.0652   1.00e-03    0.